### Preparation

In [ ]:
import os
import time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
# from mltclass import QuantumNetwork as Model
from mltclass import ClassicalNetwork as Model
from mltclass.utils import load_dataset, split_versus_dataset, split_tree_dataset, normalize_dataset, plot_history, plot_weights
from mltclass.utils import get_accuracy, get_bisection, get_leaves, get_nodes, get_tree, get_multinomial

In [ ]:
# Create folder for output data
path = "./data"
if not os.path.exists(path):
    os.makedirs(path)

In [ ]:
# Parameters
num_epochs = 2#250
num_hidden = 2#30
num_layers = 2 # Only for ClassicalNetwork
batch_size = 512
learning_rate = 0.05
rng = np.random.default_rng(2025)
torch.manual_seed(2025)
optimizer = "SGD" # Available {SGD, Adam}
balanced_dataset = False
use_bias_sigmoid = True
trainval_ratio = 0.8 # Training 80% - Validation 20%
labelmask = None # Example [0,1,5,8]
download = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Set devices
dtype = torch.float32 # Set floating point precision

In [ ]:
# Settings
dataset = "MNIST" # Available {MNIST, Fashion, CIFAR}
mode = "one_vs_rest" # Available {one_vs_rest, one_vs_one, tree}

### Dataset

In [ ]:
(X, Y), (XAll, YAll) = load_dataset(dataset, download = download, labels = labelmask)
(num_classes, num_models), (Xtrain, Ytrain), \
(Xval, Yval), (Xtest, Ytest) = split_versus_dataset(X, Y, XAll, YAll, mode, balanced_dataset, trainval_ratio, rng,\
                                                    show_population = False, device = device, dtype = dtype)
train_loader = [DataLoader(TensorDataset(X, Y), batch_size = batch_size, shuffle = True) for X,Y in zip(Xtrain, Ytrain)]
val_loader = [DataLoader(TensorDataset(X, Y), batch_size = batch_size, shuffle = False) for X,Y in zip(Xval, Yval)]

### Training

In [ ]:
start = time.perf_counter()

In [ ]:
history_train = torch.zeros((num_models,num_epochs,2), device = "cpu", dtype = torch.float32)
history_val = torch.zeros((num_models,num_epochs,2), device = "cpu", dtype = torch.float32)
hidden_weights, output_weights, models, optimizers = [], [], [], []
for idx in tqdm(range(num_models)):
    models.append(Model(Xtrain[idx].shape[1], num_hidden, use_bias_sigmoid = use_bias_sigmoid, num_layers = num_layers,\
                        device = device, dtype = dtype)) # Model
    loss_function = torch.nn.BCELoss() # Loss
    method = getattr(torch.optim, optimizer)
    optimizers.append(method(models[idx].parameters(), lr = learning_rate)) # Optimizers
    (tmptorchHid, tmptorchOut), history_train[idx,:,:], history_val[idx,:,:] = models[idx].fit(train_loader[idx],val_loader[idx],\
                                                                               num_epochs,loss_function,optimizers[idx]) # Train
    # Uncomment for (hidden and output) weights tracking
    #hidden_weights.append(tmptorchHid) 
    #output_weights.append(tmptorchOut.squeeze(0)) # Squeeze 
#hidden_weights = torch.stack(hidden_weights)
#output_weights = torch.stack(output_weights)
del tmptorchHid, tmptorchOut

In [ ]:
plot_history(history_train, history_val, show_legend = False)
plt.savefig(f"./data/{mode}_history.png")

### Inference

In [ ]:
objects = models, None, None, None # Compatibility requirement for mode = "tree"
acc = get_accuracy(objects, Xtest, Ytest, num_classes, num_models, mode, device = device, dtype = dtype) # Per-class and averaged accuracies
df = pd.DataFrame.from_dict(acc)
df = df.rename(index={df.index[-1]: 'Avg.'})
#df.style.format("{:.2f}")#.set_caption("Accuracy")
df.to_string(f"./data/{mode}_accuracy.txt")

In [ ]:
runtime = time.perf_counter() - start
print(f"Runtime of {runtime:.2f}s ({runtime/3600:.2f}h)")